<a href="https://colab.research.google.com/github/Sibu05/2693147-lab4/blob/main/Lab_1_1_Naive_Bayes_(answered).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1.1 - Naïve Bayes

## 1.1 Instructions
* Run all the code cells in chronological order.
* Put your names and student numbers in the indicated cell in the exact indicated format.
* You cannot use additional libraries to the ones imported already for you.
* Your answers must be placed in the indicated answer code cells and must follow the requested format exactly.
* Code cells containing the comment `# STUDENT ANSWER` are answer code cells.
* Do not delete the answer code cells and recreate them as they have custom metadata used by the automarker. If you accidentally delete one, you must redownload the notebook.
* **Execution Order**: Students must run all prior cells in chronological order. Specific cell dependencies will be highlighted for each task, but standard practice requires sequential execution of all `# DO NOT MODIFY` and `# STUDENT ANSWER` cells.
* **Determinism Constraint**: Students must not alter any `np.random.seed()` calls. Every call to a random function within this lab is immediately preceded by a seed to ensure identical outputs across multiple executions. Furthermore, when extracting features, you must strictly follow the ordering rules specified to ensure your matrices match the automarker's expected dimensions and order.
* Answers must be computed programmatically. Computing by hand and simply setting the variable to that value will lead to a mark of zero since these will not be valid on the private test cases.

### 1.1.1 Public Test Cases
The 'public test cases' are used for you to check your answers. Hidden test cases will be used to calculate your mark.

## 1.2 Student Details
Fill in your names and student numbers in the fields below. This is used for the automarker, so if you do this incorrectly you will receive zero as your submission will not be recognised.

Note: `student_names[0]` should correspond to the same student as `student_nos[0]` and similarly for the other students indicated.

In [3]:
# STUDENT DETAILS
student_names = ["Sibusiso Ndunge"] # List of strings containing student names
student_nos = ["2693147"] # List of strings containing student nos

assert len(student_names) == len(student_nos)

## 1.3 Imports

You are not allowed to import additional libraries to the ones specified below.

In [2]:
import numpy as np
import pandas as pd

---

## 2. Section 1: Data Preparation and the Bag of Words

In this section, we will prepare our text data using the bag of words representation. This treats each email as a long vector, with $0$ or $1$ indicating the absence or presence of a word.

### 2.0.1 Data Loading

*Requires: Cell 1.3*

In [8]:
# DO NOT MODIFY
np.random.seed(42)

# Synthetic training data (0 = Ham, 1 = Spam)
X_train_text = np.array([
    "buy this online robot",
    "send us money today",
    "send money today",
    "buy online today",
    "send us money",
    "send this money"
])
y_train = np.array([0, 1, 1, 1, 0, 1])

# Synthetic testing data
X_test_text = np.array([
    "buy money online today robot",
    "send money"
])
y_test = np.array([1, 1])

### 2.1 Question 1: Building the Vocabulary

*Requires: Cell 2.0.1*

Extract all unique words from the training dataset `X_train_text` into a single list. You must do this programmatically. Hardcoding this will lead to a mark of zero on the private test cases.

**Strict Ordering Constraint:** The vocabulary must be a Python `list`. The order of words must correspond exactly to their **first occurrence** in the training data (iterating from the first email to the last, reading left to right). Using unordered collections (like `set`) without subsequently fixing the order is strictly prohibited and will cause the automarker to fail.

Assign your resulting list to the variable `s1_q1_ans`.

In [9]:
# STUDENT ANSWER
# TODO: Create the vocabulary list and assign it to s1_q1_ans
s1_q1_ans = []
for email in X_train_text:
  word = email.split()

  for words in word:
    if (words not in s1_q1_ans):
      s1_q1_ans.append(words)

# DO NOT MODIFY BELOW
print(f"Vocabulary: {s1_q1_ans}")
if type(s1_q1_ans) is not list:
    raise TypeError("s1_q1_ans must be a list.")

Vocabulary: ['buy', 'this', 'online', 'robot', 'send', 'us', 'money', 'today']


#### 2.1.1 Public Test Case

In [12]:
# DO NOT MODIFY
ptc_s1_q1 = ['buy', 'this', 'online', 'robot', 'send', 'us', 'money', 'today']

if s1_q1_ans == ptc_s1_q1:
    print("S1 Q1 Public Test Case Passed")
else:
    print("S1 Q1 Public Test Case Failed")

S1 Q1 Public Test Case Passed


### 2.2 Question 2: Vectorisation

*Requires: Cells 2.0.1 and 2.1*

In the lecture, we introduced the "Bag of words representation", which allows us to treat "each email as a long vector, with $0/1$ indicating absence/presence of a word". We defined this "vector of attributes/features" as $x=(x_{1},x_{2},...,x_{n})$.

When working with a complete dataset, it is standard practice to stack these individual feature vectors row by row to form a single 2D array. This resulting 2D structure—where each row represents a single observation (an email) and each column represents a specific feature (a word in your vocabulary)—is commonly referred to as a **design matrix**.

Convert the text emails in `X_train_text` into a design matrix consisting of binary feature vectors indicating the presence ($1$) or absence ($0$) of words.

**Strict Ordering Constraint:** The feature at index $i$ in your resulting vector must correspond to the word at index $i$ in the vocabulary list generated in Q1.

Assign your resulting design matrix to the variable `s1_q2_ans`.

In [10]:
# STUDENT ANSWER
# TODO: Create the design matrix for the training data and assign it to s1_q2_ans
s1_q2_ans = np.zeros((len(X_train_text), len(s1_q1_ans)), dtype=float)
for i, email in enumerate(X_train_text):
    words = email.split()

    for j, vocab_word in enumerate(s1_q1_ans):
        if vocab_word in words:
            s1_q2_ans[i][j] = 1

# DO NOT MODIFY BELOW
print(f"Shape of s1_q2_ans: {s1_q2_ans.shape}")

Shape of s1_q2_ans: (6, 8)


#### 2.2.1 Public Test Case

In [11]:
# DO NOT MODIFY
ptc_s1_q2_row0 = np.array([1., 1., 1., 1., 0., 0., 0., 0.])

if np.array_equal(s1_q2_ans[0], ptc_s1_q2_row0):
    print("S1 Q2 Public Test Case Passed")
else:
    print("S1 Q2 Public Test Case Failed")

S1 Q2 Public Test Case Passed


---

## 3. Section 2: Learning the Naïve Bayes Classifier

### 3.1 Question 1: Class Priors

*Requires: Cell 2.0.1*

Compute the prior probability of each class, $P(c)$, based on the training data `y_train`. Some classes may be more common than others.

Assign your numpy array containing `[P(ham), P(spam)]` to the variable `s2_q1_ans`.

In [13]:
# STUDENT ANSWER
# TODO: Calculate [P(ham), P(spam)] and assign it to s2_q1_ans
s2_q1_ans = np.zeros(2, dtype=float)
total = len(y_train)

s2_q1_ans[0] = np.count_nonzero(y_train == 0) / total
s2_q1_ans[1] = np.count_nonzero(y_train == 1) / total

# DO NOT MODIFY BELOW
print(f"Priors: {s2_q1_ans}")

Priors: [0.33333333 0.66666667]


#### 3.1.1 Public Test Case

In [14]:
# DO NOT MODIFY
ptc_s2_q1 = np.array([0.333, 0.667])

if np.array_equal(np.round(s2_q1_ans, 3), ptc_s2_q1):
    print("S2 Q1 Public Test Case Passed")
else:
    print("S2 Q1 Public Test Case Failed")

S2 Q1 Public Test Case Passed


### 3.2 Question 2: Class Conditional Probabilities (Unsmoothed)

*Requires: Cells 2.0.1 and 2.2*

Compute the class conditional models $P(x_{i}|c)$ for all features given each class. This defines how likely class $c$ is to generate the observation $x_{i}$.

Create a Numpy array of shape `(2, V)`, where row 0 represents Ham, row 1 represents Spam, and $V$ is the length of your vocabulary.

Assign your numpy array to the variable `s2_q2_ans`.

In [15]:
# STUDENT ANSWER
# TODO: Calculate class conditional probabilities and assign to s2_q2_ans
V = len(s1_q1_ans)

s2_q2_ans = np.zeros((2, V), dtype=float)

ham_rows = s1_q2_ans[y_train == 0]
spam_rows = s1_q2_ans[y_train == 1]

s2_q2_ans[0] = np.mean(ham_rows, axis=0)
s2_q2_ans[1] = np.mean(spam_rows, axis=0)


# DO NOT MODIFY BELOW
print(f"Conditionals shape: {s2_q2_ans.shape}")

Conditionals shape: (2, 8)


#### 3.2.1 Public Test Case

In [16]:
# DO NOT MODIFY
ptc_s2_q2_slice = np.array([
    [0.5 , 0.5 , 0.5 , 0.5 ],
    [0.25, 0.25, 0.25, 0.0 ]
])

if np.array_equal(np.round(s2_q2_ans[:, :4], 3), ptc_s2_q2_slice):
    print("S2 Q2 Public Test Case Passed")
else:
    print("S2 Q2 Public Test Case Failed")

S2 Q2 Public Test Case Passed


---

## 4. Section 3: Inference and Smoothing

### 4.1 Question 1: Inference (The Zero-Frequency Problem)

*Requires: Cells 2.1, 2.2, 3.1, and 3.2*

Calculate the posterior probability for the first test email (`X_test_text[0]`) using the unsmoothed model.

Use the Naïve Bayes model formulation applied to Bayes' rule:


$$P(c|x)\propto P(c)\prod_{i=1}^{n}P(x_{i}|c)$$

Note: You must first vectorise the test email using your vocabulary from Section 1.

Assign your predicted probability (the highest unnormalised posterior probability) to the variable `s3_q1_ans`.

In [19]:
# STUDENT ANSWER
# TODO: Calculate the unnormalised posterior probability for both classes, then return the highest.
# Hint: You are predicting the class for "buy money online today robot"
x = np.zeros(len(s1_q1_ans))

for i, word in enumerate(s1_q1_ans):
    if word in "buy money online today robot".split():
        x[i] = 1
posteriors = np.zeros(2)

for c in range(2):
    prob = s2_q1_ans[c]

    for i in range(len(x)):
        if x[i] == 1:
            prob *= s2_q2_ans[c][i]
        else:
            prob *= (1 - s2_q2_ans[c][i])

    posteriors[c] = prob
s3_q1_ans = np.max(posteriors)

# DO NOT MODIFY BELOW
print(f"Predicted Probability: {s3_q1_ans}")

Predicted Probability: 0.0


#### 4.1.1 Public Test Case

In [20]:
# DO NOT MODIFY
if s3_q1_ans == 0.0:
    print("S3 Q1 Public Test Case Passed")
else:
    print("S3 Q1 Public Test Case Failed")

S3 Q1 Public Test Case Passed


### 4.2 Question 2: Understanding the Problem

*Requires: Cell 4.1*

Why did the calculation in Question 1 evaluate to zero?

* a) The email is definitely not spam.
* b) The zero-frequency problem occurred because a word in the test email never appeared in the training data.
* c) The class priors were zero.
* d) The independence assumption failed.

Assign your chosen letter as a string to the variable `s3_q2_ans`.

In [21]:
# STUDENT ANSWER
s3_q2_ans = "b" # Set your answer here

# DO NOT MODIFY BELOW
if s3_q2_ans in ["a", "b", "c", "d"]:
    print("s3_q2_ans is in the correct format")
else:
    print("s3_q2_ans is NOT in the correct format")

s3_q2_ans is in the correct format


### 4.3 Question 3: Laplace Smoothing

*Requires: Cells 2.2 and 3.1*

Never allow zero probabilities. Re-calculate the class conditional probabilities using Laplace smoothing: add a small positive number to each count.

Use the following deterministic formula where $k=1$:


$$P(x_{i}=1|c)=\frac{N_{x_{i}=1,c}+1}{N_{c}+2}$$

Assign your smoothed numpy array to the variable `s3_q3_ans`.

### 4.3 Question 3: Laplace Smoothing

*Requires: Cells 2.2 and 3.1*

Never allow zero probabilities. Re-calculate the class conditional probabilities using Laplace smoothing: add a small positive number to each count.

Use the following deterministic formula where $k=1$:


$$P(x_{i}=1|c)=\frac{N_{x_{i}=1,c}+1}{N_{c}+2}$$

Assign your smoothed numpy array to the variable `s3_q3_ans`.

In [25]:
# STUDENT ANSWER
# TODO: Recalculate smoothed probabilities and assign to s3_q3_ans
V = len(s1_q1_ans)
s3_q3_ans = np.zeros((2, V), dtype=float)

for c in [0, 1]:
    X_c = s1_q2_ans[y_train == c]
    Nc = X_c.shape[0]
    word_counts_c = np.sum(X_c, axis=0)

    s3_q3_ans[c] = (word_counts_c + 1) / (Nc + 2)


# DO NOT MODIFY BELOW
print(f"Smoothed Conditionals shape: {s3_q3_ans.shape}")

Smoothed Conditionals shape: (2, 8)


#### 4.3.1 Public Test Case

In [26]:
# DO NOT MODIFY
ptc_s3_q3_slice = np.array([
    [0.5  , 0.5  , 0.5  , 0.5  ],
    [0.333, 0.333, 0.333, 0.167]
])

if np.array_equal(np.round(s3_q3_ans[:, :4], 3), ptc_s3_q3_slice):
    print("S3 Q3 Public Test Case Passed")
else:
    print("S3 Q3 Public Test Case Failed")

S3 Q3 Public Test Case Passed


### 4.4 Question 4: Smoothed Inference

*Requires: Cells 3.1 and 4.3*

Re-run the inference from Question 1 using the smoothed model (`s3_q3_ans`). Predict the class for `X_test_text[0]`. In the event of a tie, use `np.argmax()` to deterministically select the first index.

Assign your predicted class integer (0 for ham, 1 for spam) to the variable `s3_q4_ans`.

In [28]:
# STUDENT ANSWER
# TODO: Predict the class and assign it to s3_q4_ans
# Vectorize the test email
x_test = np.zeros(len(s1_q1_ans))
test_words = "buy money online today robot".split()
for i, word in enumerate(s1_q1_ans):
    if word in test_words:
        x_test[i] = 1

posteriors = np.zeros(2)
for c in range(2):
    prob = s2_q1_ans[c]
    for i in range(len(x_test)):
        if x_test[i] == 1:
            prob *= s3_q3_ans[c][i]
        else:
            prob *= (1 - s3_q3_ans[c][i])
    posteriors[c] = prob

s3_q4_ans = int(np.argmax(posteriors))

# DO NOT MODIFY BELOW
print(f"Predicted Class: {s3_q4_ans}")

Predicted Class: 1


#### 4.4.1 Public Test Case

In [29]:
# DO NOT MODIFY
if s3_q4_ans == 1:
    print("S3 Q4 Public Test Case Passed")
else:
    print("S3 Q4 Public Test Case Failed")

S3 Q4 Public Test Case Passed


---

## 5. Section 4: Model Evaluation

### 5.1 Question 1: Predictions on Test Set

*Requires: Cells 2.1 and 4.3*

Generate predictions for the entire `X_test_text` dataset using the smoothed model.

Assign your 1D array of predicted labels to the variable `s4_q1_ans`.

In [30]:
# STUDENT ANSWER
# TODO: Generate predictions and assign to s4_q1_ans
s4_q1_ans = np.zeros(len(X_test_text), dtype=int)

for i, email in enumerate(X_test_text):

    x_test_vec = np.zeros(len(s1_q1_ans))
    email_words = email.split()
    for j, vocab_word in enumerate(s1_q1_ans):
        if vocab_word in email_words:
            x_test_vec[j] = 1

    posteriors = np.zeros(2)
    for c in range(2):
        prob = s2_q1_ans[c]
        for k in range(len(x_test_vec)):
            if x_test_vec[k] == 1:
                prob *= s3_q3_ans[c][k]
            else:
                prob *= (1 - s3_q3_ans[c][k])
        posteriors[c] = prob

    s4_q1_ans[i] = np.argmax(posteriors)

# DO NOT MODIFY BELOW
print(f"Predictions: {s4_q1_ans}")

Predictions: [1 1]


#### 5.1.1 Public Test Case

In [31]:
# DO NOT MODIFY
ptc_s4_q1 = np.array([1, 1])

if np.array_equal(s4_q1_ans, ptc_s4_q1):
    print("S4 Q1 Public Test Case Passed")
else:
    print("S4 Q1 Public Test Case Failed")

S4 Q1 Public Test Case Passed


### 5.2 Question 2: Accuracy

*Requires: Cell 5.1*

Compute the overall accuracy of the model on the test set.

$$\text{Accuracy}=\frac{\text{Correct Predictions}}{\text{Total Predictions}}$$

Assign your calculated accuracy float value to the variable `s4_q2_ans`.

In [32]:
# STUDENT ANSWER
# TODO: Calculate accuracy and assign it to s4_q2_ans
correct_predictions = np.sum(s4_q1_ans == y_test)
total_predictions = len(y_test)
s4_q2_ans = float(correct_predictions / total_predictions)

# DO NOT MODIFY BELOW
print(f"Accuracy: {s4_q2_ans}")

Accuracy: 1.0


#### 5.2.1 Public Test Case

In [33]:
# DO NOT MODIFY
if np.round(s4_q2_ans, 3) == 1.000:
    print("S4 Q2 Public Test Case Passed")
else:
    print("S4 Q2 Public Test Case Failed")

S4 Q2 Public Test Case Passed


# END